# 03 BIDMC Baseline Logistic Regression

Train grouped logistic-regression baselines on canonical BIDMC windows with split diagnostics and threshold evaluation.

Inputs from: `outputs/bidmc_window_features.csv` (from notebook 02).
Outputs to: printed baseline diagnostics for BIDMC validation evidence.
Next: notebook 05 to continue OpenOximetry main-dataset processing.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMN_NAMES = (
    'mean','std','minimum','maximum','value_range','rms','skewness','kurtosis','energy',
    'derivative_std','dominant_frequency_hz','bandpower_0_5_2_hz','bandpower_2_5_hz','bandpower_ratio_0_5_2_to_2_5'
)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').is_dir() and (candidate / 'outputs').is_dir():
            return candidate
    raise RuntimeError('Could not find project root containing notebooks/ and outputs/')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ARTIFACT_FEATURE_CSV = PROJECT_ROOT / 'outputs' / 'bidmc_window_features.csv'


## Load and Prepare Data


In [ ]:
seed = 42
thresholds = (0.2, 0.3, 0.4, 0.5, 0.6)

if not ARTIFACT_FEATURE_CSV.is_file():
    raise FileNotFoundError(
        f"Feature CSV was not found at {ARTIFACT_FEATURE_CSV}. Run notebook 02 first."
    )

# Load feature rows from CSV
with ARTIFACT_FEATURE_CSV.open(newline="", encoding="utf-8") as handle:
    feature_rows = list(csv.DictReader(handle))

feature_source = str(ARTIFACT_FEATURE_CSV)
print(f"Loaded {len(feature_rows)} feature rows from {feature_source}")


### Build Feature Matrix and Labels

Convert loaded CSV rows into numpy arrays for the feature matrix, labels, and group identifiers. This prepares the data for model training and evaluation.

In [ ]:
# Build feature matrix and labels
feature_matrix = np.asarray(
    [[float(row[fname]) for fname in FEATURE_COLUMN_NAMES] for row in feature_rows],
    dtype=float,
)
labels = np.asarray([int(row["label"]) for row in feature_rows], dtype=int)
groups = np.asarray([str(row["recording_id"]) for row in feature_rows], dtype=str)

unique_groups = sorted(set(groups.tolist()))
positive_groups = sorted(set(groups[labels == 1].tolist()))

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Positive labels: {labels.sum()} / {len(labels)}")
print(f"Unique groups: {len(unique_groups)}")
print(f"Positive groups: {len(positive_groups)} / {len(unique_groups)}")
print(f"Positive group IDs: {', '.join(positive_groups)}")
print()


### Generate Valid Group Splits

Use GroupShuffleSplit to create train/test splits while ensuring no duplicate group pairs and both splits contain both positive and negative examples. Two split sets are tracked:
- Primary grouped splits: validity-filtered grouped resamples (can reuse the same positive test recordings across folds)
- Strict grouped splits: enforce non-repeated positive test recordings across reported folds (may yield fewer folds when positives are scarce)

In [ ]:
# Generate valid group splits with GroupShuffleSplit
n_splits = 5
test_size = 0.2
max_attempts = 400
min_strict_splits_for_aggregate = 3

splitter = GroupShuffleSplit(
    n_splits=max_attempts,
    test_size=test_size,
    random_state=seed,
)
placeholder = np.zeros(labels.shape[0], dtype=float)

valid_splits = []
seen_group_pairs = set()

for train_indices, test_indices in splitter.split(placeholder, labels, groups):
    train_groups = tuple(sorted(set(groups[train_indices].tolist())))
    test_groups = tuple(sorted(set(groups[test_indices].tolist())))
    split_key = (train_groups, test_groups)

    if split_key in seen_group_pairs:
        continue
    seen_group_pairs.add(split_key)

    # Check for binary classification in both splits
    if len(np.unique(labels[train_indices])) < 2:
        continue
    if len(np.unique(labels[test_indices])) < 2:
        continue

    valid_splits.append((train_indices, test_indices))
    if len(valid_splits) >= n_splits:
        break

# Strict split mode: do not reuse positive test recordings across accepted folds.
strict_valid_splits = []
seen_strict_group_pairs = set()
used_strict_test_positive_groups = set()

for train_indices, test_indices in splitter.split(placeholder, labels, groups):
    train_groups = tuple(sorted(set(groups[train_indices].tolist())))
    test_groups = tuple(sorted(set(groups[test_indices].tolist())))
    split_key = (train_groups, test_groups)

    if split_key in seen_strict_group_pairs:
        continue
    seen_strict_group_pairs.add(split_key)

    if len(np.unique(labels[train_indices])) < 2:
        continue
    if len(np.unique(labels[test_indices])) < 2:
        continue

    test_positive_group_set = set(groups[test_indices][labels[test_indices] == 1].tolist())
    if not test_positive_group_set:
        continue
    if used_strict_test_positive_groups.intersection(test_positive_group_set):
        continue

    strict_valid_splits.append((train_indices, test_indices))
    used_strict_test_positive_groups.update(test_positive_group_set)

    if len(strict_valid_splits) >= n_splits:
        break

print(f"Generated {len(valid_splits)} primary valid splits")
print(f"Generated {len(strict_valid_splits)} strict valid splits (no repeated positive test recordings)")
if len(strict_valid_splits) < min_strict_splits_for_aggregate:
    print(
        "Strict mode warning: too few strict splits for robust aggregate reporting; "
        "strict results will be treated as fold-level diagnostics."
    )
print()


### Train Baseline Logistic Regression with Nested Threshold Selection

For each grouped outer split: scale features, train logistic regression, tune threshold on inner grouped validation folds, and evaluate once on held-out groups.

Leakage guard: this design preserves recording-level separation between train and test at every outer split.

In [ ]:
# ===== TRAIN AND EVALUATE LOGISTIC REGRESSION BASELINE =====
# Evaluate two analysis modes:
#   - primary_grouped: valid grouped splits (can reuse positive test recordings across folds)
#   - strict_unique_positive_test: grouped splits that do not reuse positive test recordings
#
# Inner threshold tuning uses a shared threshold grid for all inner folds so cross-fold scores
# are directly comparable.

inner_threshold_grid = np.round(np.linspace(0.1, 0.9, 17), 2)


def evaluate_split_collection(split_collection, analysis_mode):
    split_results_local = []
    seen_test_positive_group_ids = set()

    for split_index, (train_indices, test_indices) in enumerate(split_collection, start=1):
        # ===== EXTRACT TRAIN/TEST INDICES AND METADATA =====
        x_train_raw = feature_matrix[train_indices]
        x_test_raw = feature_matrix[test_indices]
        y_train = labels[train_indices]
        y_test = labels[test_indices]
        train_groups = sorted(set(groups[train_indices].tolist()))
        test_groups = sorted(set(groups[test_indices].tolist()))
        train_positive_groups = sorted(set(groups[train_indices][labels[train_indices] == 1].tolist()))
        test_positive_groups = sorted(set(groups[test_indices][labels[test_indices] == 1].tolist()))

        # Track positive-test overlap across splits for caveat reporting.
        repeated_test_positive_groups = sorted(
            set(test_positive_groups).intersection(seen_test_positive_group_ids)
        )
        seen_test_positive_group_ids.update(test_positive_groups)

        # ===== NORMALIZE FEATURES =====
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train_raw)
        x_test = scaler.transform(x_test_raw)

        # ===== TRAIN OUTER MODEL =====
        model = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            solver="liblinear",
            random_state=seed,
        )
        model.fit(x_train, y_train)

        # ===== INITIALIZE THRESHOLD TUNING VARIABLES =====
        best_threshold = 0.5
        best_inner_f1 = float("nan")
        threshold_selection_method = "fixed_0.5_fallback"
        threshold_fallback_reason = "inner_tuning_not_attempted"
        inner_valid_split_count = 0
        selected_threshold_support = 0

        # ===== INNER GROUPED THRESHOLD TUNING (OPTIONAL) =====
        if len(np.unique(y_train)) >= 2:
            threshold_fallback_reason = "inner_tuning_no_valid_inner_folds"
            inner_groups = groups[train_indices]
            inner_splitter = GroupShuffleSplit(
                n_splits=20,
                test_size=0.3,
                random_state=seed + split_index,
            )
            inner_threshold_scores = {float(t): [] for t in inner_threshold_grid.tolist()}
            seen_inner_group_pairs = set()

            for inner_train_rel, inner_val_rel in inner_splitter.split(x_train_raw, y_train, groups=inner_groups):
                inner_train_group_ids = tuple(sorted(set(inner_groups[inner_train_rel].tolist())))
                inner_val_group_ids = tuple(sorted(set(inner_groups[inner_val_rel].tolist())))
                inner_split_key = (inner_train_group_ids, inner_val_group_ids)
                if inner_split_key in seen_inner_group_pairs:
                    continue
                seen_inner_group_pairs.add(inner_split_key)

                y_inner_train = y_train[inner_train_rel]
                y_inner_val = y_train[inner_val_rel]
                if len(np.unique(y_inner_train)) < 2 or len(np.unique(y_inner_val)) < 2:
                    continue

                inner_scaler = StandardScaler()
                x_inner_train = inner_scaler.fit_transform(x_train_raw[inner_train_rel])
                x_inner_val = inner_scaler.transform(x_train_raw[inner_val_rel])

                inner_model = LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    solver="liblinear",
                    random_state=seed,
                )
                inner_model.fit(x_inner_train, y_inner_train)
                inner_val_probs = inner_model.predict_proba(x_inner_val)[:, 1]

                for threshold in inner_threshold_grid:
                    preds = (inner_val_probs >= threshold).astype(int)
                    f1 = float(f1_score(y_inner_val, preds, zero_division=0))
                    inner_threshold_scores[float(threshold)].append(f1)

                inner_valid_split_count += 1

            if inner_valid_split_count > 0:
                threshold_rankings = [
                    (threshold, float(np.mean(scores)), len(scores))
                    for threshold, scores in inner_threshold_scores.items()
                    if len(scores) > 0
                ]
                threshold_rankings.sort(key=lambda item: (-item[1], abs(item[0] - 0.5), item[0]))
                best_threshold, best_inner_f1, selected_threshold_support = threshold_rankings[0]
                threshold_selection_method = "nested_inner_group_validation_f1_grid"
                threshold_fallback_reason = "none"
        else:
            threshold_fallback_reason = "outer_train_single_class"

        # ===== EVALUATE ON OUTER TEST FOLD =====
        test_probs = model.predict_proba(x_test)[:, 1]
        test_preds = (test_probs >= best_threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, test_preds, labels=[0, 1]).ravel()

        split_results_local.append({
            "analysis_mode": analysis_mode,
            "split_index": split_index,
            "train_group_count": len(train_groups),
            "test_group_count": len(test_groups),
            "train_groups": ";".join(train_groups),
            "test_groups": ";".join(test_groups),
            "train_positive_group_count": len(train_positive_groups),
            "test_positive_group_count": len(test_positive_groups),
            "train_positive_groups": ";".join(train_positive_groups),
            "test_positive_groups": ";".join(test_positive_groups),
            "reused_test_positive_group_count": len(repeated_test_positive_groups),
            "reused_test_positive_groups": ";".join(repeated_test_positive_groups),
            "train_positive_row_count": int(y_train.sum()),
            "test_positive_row_count": int(y_test.sum()),
            "predicted_positive_row_count": int(test_preds.sum()),
            "decision_threshold": float(best_threshold),
            "threshold_selection_method": threshold_selection_method,
            "threshold_fallback_reason": threshold_fallback_reason,
            "threshold_tuning_inner_split_count": int(inner_valid_split_count),
            "selected_threshold_support": int(selected_threshold_support),
            "selected_threshold_inner_mean_f1": float(best_inner_f1) if np.isfinite(best_inner_f1) else float("nan"),
            "accuracy": float(accuracy_score(y_test, test_preds)),
            "precision": float(precision_score(y_test, test_preds, zero_division=0)),
            "recall": float(recall_score(y_test, test_preds, zero_division=0)),
            "f1": float(f1_score(y_test, test_preds, zero_division=0)),
            "true_negative": int(tn),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_positive": int(tp),
        })

    return split_results_local


primary_split_results = evaluate_split_collection(valid_splits, "primary_grouped")
strict_split_results = evaluate_split_collection(strict_valid_splits, "strict_unique_positive_test")

# Keep backward compatibility for downstream cells that still read split_results.
split_results = primary_split_results

print(f"Primary split results: {len(primary_split_results)}")
print(f"Strict split results: {len(strict_split_results)}")
print()


### Aggregate Baseline Results

Combine metrics across valid grouped resamples. Threshold statistics are reported as exploratory diagnostics only (not a robust deployment operating point), because rare-class support is extremely limited per split.

In [ ]:
# ===== AGGREGATE BASELINE RESULTS ACROSS ANALYSIS MODES =====
# Summarize primary grouped results and strict unique-positive-test results separately.

positive_recordings = sorted(set(groups[labels == 1].tolist()))


def build_mode_summary(mode_name, mode_split_results):
    if not mode_split_results:
        return {
            "analysis_mode": mode_name,
            "model_name": "logreg_default",
            "feature_source": feature_source,
            "sample_count": feature_matrix.shape[0],
            "positive_row_count": int(labels.sum()),
            "positive_recording_count": len(positive_recordings),
            "positive_recordings": ";".join(positive_recordings),
            "recording_count": len(set(groups.tolist())),
            "valid_split_count": 0,
            "nested_threshold_split_count": 0,
            "fallback_threshold_split_count": 0,
            "splits_with_reused_test_positive_groups": 0,
            "distinct_test_positive_group_sets": 0,
            "exploratory_mean_split_threshold": float("nan"),
            "exploratory_std_split_threshold": float("nan"),
            "nested_only_mean_threshold": float("nan"),
            "nested_only_std_threshold": float("nan"),
            "reporting_fixed_threshold": 0.5,
            "mean_accuracy": float("nan"),
            "std_accuracy": float("nan"),
            "mean_precision": float("nan"),
            "std_precision": float("nan"),
            "mean_recall": float("nan"),
            "std_recall": float("nan"),
            "mean_f1": float("nan"),
            "std_f1": float("nan"),
        }

    thresholds_arr = np.asarray([r["decision_threshold"] for r in mode_split_results], dtype=float)
    accuracies = np.asarray([r["accuracy"] for r in mode_split_results], dtype=float)
    precisions = np.asarray([r["precision"] for r in mode_split_results], dtype=float)
    recalls = np.asarray([r["recall"] for r in mode_split_results], dtype=float)
    f1s = np.asarray([r["f1"] for r in mode_split_results], dtype=float)

    selection_methods = [r["threshold_selection_method"] for r in mode_split_results]
    nested_count = int(sum(1 for method in selection_methods if method == "nested_inner_group_validation_f1_grid"))
    fallback_count = int(sum(1 for method in selection_methods if method == "fixed_0.5_fallback"))

    nested_thresholds = np.asarray(
        [
            r["decision_threshold"]
            for r in mode_split_results
            if r["threshold_selection_method"] == "nested_inner_group_validation_f1_grid" and np.isfinite(r["decision_threshold"])
        ],
        dtype=float,
    )

    test_positive_group_sets = sorted(
        set(r["test_positive_groups"] for r in mode_split_results)
    )

    return {
        "analysis_mode": mode_name,
        "model_name": "logreg_default",
        "feature_source": feature_source,
        "sample_count": feature_matrix.shape[0],
        "positive_row_count": int(labels.sum()),
        "positive_recording_count": len(positive_recordings),
        "positive_recordings": ";".join(positive_recordings),
        "recording_count": len(set(groups.tolist())),
        "valid_split_count": len(mode_split_results),
        "nested_threshold_split_count": nested_count,
        "fallback_threshold_split_count": fallback_count,
        "splits_with_reused_test_positive_groups": int(
            sum(1 for r in mode_split_results if r["reused_test_positive_group_count"] > 0)
        ),
        "distinct_test_positive_group_sets": len(test_positive_group_sets),
        "exploratory_mean_split_threshold": float(np.mean(thresholds_arr)) if thresholds_arr.size else float("nan"),
        "exploratory_std_split_threshold": float(np.std(thresholds_arr)) if thresholds_arr.size else float("nan"),
        "nested_only_mean_threshold": float(np.mean(nested_thresholds)) if nested_thresholds.size else float("nan"),
        "nested_only_std_threshold": float(np.std(nested_thresholds)) if nested_thresholds.size else float("nan"),
        "reporting_fixed_threshold": 0.5,
        "mean_accuracy": float(np.mean(accuracies)),
        "std_accuracy": float(np.std(accuracies)),
        "mean_precision": float(np.mean(precisions)),
        "std_precision": float(np.std(precisions)),
        "mean_recall": float(np.mean(recalls)),
        "std_recall": float(np.std(recalls)),
        "mean_f1": float(np.mean(f1s)),
        "std_f1": float(np.std(f1s)),
    }


baseline_summary_primary = build_mode_summary("primary_grouped", primary_split_results)
baseline_summary_strict = build_mode_summary("strict_unique_positive_test", strict_split_results)
baseline_summary = baseline_summary_primary
strict_has_aggregate_support = len(strict_split_results) >= min_strict_splits_for_aggregate

print("=== Baseline Logistic Regression Results: Primary Grouped Mode ===")
for key, val in baseline_summary_primary.items():
    if key != "feature_source":
        print(f"{key}: {val}")
print()

print("=== Baseline Logistic Regression Results: Strict Unique-Positive Mode ===")
for key, val in baseline_summary_strict.items():
    if key != "feature_source":
        print(f"{key}: {val}")
print()

print(
    "Threshold note: per-split tuned thresholds are exploratory diagnostics only. "
    "Use the fixed reporting threshold (0.5) as the nominal operating point unless "
    "a separate external calibration set is available."
)
print()

if baseline_summary_primary["fallback_threshold_split_count"] > 0:
    print(
        "Primary mode caveat: one or more splits used fixed 0.5 fallback because nested inner tuning "
        "did not have enough valid inner folds."
    )
    print()

if baseline_summary_primary["splits_with_reused_test_positive_groups"] > 0:
    print(
        "Primary mode caveat: positive test recordings repeat across some folds, so this is not an "
        "independent rare-class validation." 
    )
    print()

if not strict_has_aggregate_support:
    print(
        "Strict mode caveat: fewer than the minimum strict folds were available; treat strict-mode outputs "
        "as fold-level diagnostics rather than robust aggregate evidence."
    )
    print()


### Analyze Threshold Sweep Across Splits

Test fixed thresholds (0.2, 0.3, 0.4, 0.5, 0.6) across all splits to analyze performance trade-offs between sensitivity, specificity, precision, and recall.

In [ ]:
# ===== THRESHOLD SWEEP ANALYSIS =====
# This cell re-trains models on each outer split and evaluates them at fixed predefined thresholds
# (0.2, 0.3, 0.4, 0.5, 0.6). This provides exploratory analysis of performance trade-offs across
# operating points without the bias from threshold tuning on the same data.

# Initialize dictionary to collect metrics for each threshold
per_threshold_metrics = {t: [] for t in thresholds}

# ===== TRAIN MODELS AND COLLECT PER-THRESHOLD METRICS =====
# For each valid outer split, train a fresh model and evaluate it across all thresholds
for train_indices, test_indices in valid_splits:
    # Normalize features (fit on train, apply to test)
    scaler = StandardScaler()
    x_train = scaler.fit_transform(feature_matrix[train_indices])
    x_test = scaler.transform(feature_matrix[test_indices])
    y_train = labels[train_indices]
    y_test = labels[test_indices]
    
    # Train logistic regression model
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="liblinear",
        random_state=seed,
    )
    model.fit(x_train, y_train)
    test_probs = model.predict_proba(x_test)[:, 1]
    
    # ===== EVALUATE EACH THRESHOLD =====
    # For every predefined threshold, compute predictions and metrics
    for threshold in thresholds:
        preds = (test_probs >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
        per_threshold_metrics[threshold].append({
            "accuracy": float(accuracy_score(y_test, preds)),
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall": float(recall_score(y_test, preds, zero_division=0)),
            "f1": float(f1_score(y_test, preds, zero_division=0)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })

# ===== AGGREGATE METRICS ACROSS SPLITS PER THRESHOLD =====
# For each threshold, compute mean and std of metrics across all splits
sweep_results = []
for threshold in thresholds:
    metrics_list = per_threshold_metrics[threshold]
    
    # Extract metric arrays for aggregation
    accs = np.asarray([m["accuracy"] for m in metrics_list], dtype=float)
    precs = np.asarray([m["precision"] for m in metrics_list], dtype=float)
    recs = np.asarray([m["recall"] for m in metrics_list], dtype=float)
    f1_vals = np.asarray([m["f1"] for m in metrics_list], dtype=float)
    
    # Compute aggregated result row
    sweep_results.append({
        "threshold": threshold,
        "split_count": len(metrics_list),
        "mean_accuracy": float(np.mean(accs)),
        "std_accuracy": float(np.std(accs)),
        "mean_precision": float(np.mean(precs)),
        "std_precision": float(np.std(precs)),
        "mean_recall": float(np.mean(recs)),
        "std_recall": float(np.std(recs)),
        "mean_f1": float(np.mean(f1_vals)),
        "std_f1": float(np.std(f1_vals)),
        "total_tn": int(sum(m["tn"] for m in metrics_list)),
        "total_fp": int(sum(m["fp"] for m in metrics_list)),
        "total_fn": int(sum(m["fn"] for m in metrics_list)),
        "total_tp": int(sum(m["tp"] for m in metrics_list)),
    })


### Display Results

Print formatted tables of baseline summary metrics and threshold sweep analysis for visual review.

In [ ]:
# Display summary tables
summary_table = pd.DataFrame([baseline_summary_primary, baseline_summary_strict])
print("=== Baseline Summaries (Primary + Strict) ===")
print(summary_table.to_string(index=False))
print()

primary_split_df = pd.DataFrame(primary_split_results).sort_values("split_index").reset_index(drop=True)
print("=== Primary Mode Per-Split Allocation, Thresholding, And Confusion ===")
if not primary_split_df.empty:
    print(
        primary_split_df[
            [
                "split_index",
                "train_group_count",
                "test_group_count",
                "train_positive_group_count",
                "test_positive_group_count",
                "reused_test_positive_group_count",
                "train_positive_row_count",
                "test_positive_row_count",
                "threshold_selection_method",
                "threshold_fallback_reason",
                "threshold_tuning_inner_split_count",
                "selected_threshold_support",
                "decision_threshold",
                "predicted_positive_row_count",
                "true_negative",
                "false_positive",
                "false_negative",
                "true_positive",
                "accuracy",
                "precision",
                "recall",
                "f1",
            ]
        ].to_string(index=False)
    )
print()

print("=== Primary Mode Test Positive Group IDs Per Split ===")
if not primary_split_df.empty:
    print(primary_split_df[["split_index", "test_positive_groups", "reused_test_positive_groups"]].to_string(index=False))
print()

strict_split_df = pd.DataFrame(strict_split_results).sort_values("split_index").reset_index(drop=True)
print("=== Strict Mode Per-Split Allocation, Thresholding, And Confusion ===")
if not strict_split_df.empty:
    print(
        strict_split_df[
            [
                "split_index",
                "train_group_count",
                "test_group_count",
                "train_positive_group_count",
                "test_positive_group_count",
                "reused_test_positive_group_count",
                "train_positive_row_count",
                "test_positive_row_count",
                "threshold_selection_method",
                "threshold_fallback_reason",
                "threshold_tuning_inner_split_count",
                "selected_threshold_support",
                "decision_threshold",
                "predicted_positive_row_count",
                "true_negative",
                "false_positive",
                "false_negative",
                "true_positive",
                "accuracy",
                "precision",
                "recall",
                "f1",
            ]
        ].to_string(index=False)
    )
else:
    print("No strict splits were available under the unique-positive-test constraint.")
print()

print("=== Strict Mode Test Positive Group IDs Per Split ===")
if not strict_split_df.empty:
    print(strict_split_df[["split_index", "test_positive_groups", "reused_test_positive_groups"]].to_string(index=False))
print()

# Display threshold sweep results (computed on primary grouped splits)
sweep_df = pd.DataFrame(sweep_results)
print("=== Threshold Sweep Results (Fixed, Predefined Thresholds; Primary Mode) ===")
print(sweep_df[["threshold", "mean_accuracy", "mean_precision", "mean_recall", "mean_f1"]].to_string(index=False))


## Run Baseline Evaluation and Threshold Sweep
